In [ ]:
#Imports
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.openai import OpenAIEmbedding
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from llama_index.core import Document
from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI
import faiss
import os
import re

In [ ]:
"""
Summary:
Initializes key data structures for tracking user learning progress and preferences.
The `topic` dictionary stores the user's performance or familiarity level across core computer science topics. The `user_profile` dictionary aggregates this data 
along with the user’s learning style and prior knowledge parameter.

Remarks:
- Each key in `topic` starts with a value of 0, representing a neutral or uninitialized state.
- `learning_style` can later hold categorical information such as 'action-based', relationship-based', or 'mixed'.
- `prior_A` can represent prior probability or initial belief (useful in Bayesian updating).
- The `topic` dictionary is nested directly inside `user_profile` for easy centralized access.
"""

topic = {
    "array": 0,
    "dictionary" : 0,
    "linked list": 0,
    "tree": 0,
    "graph": 0
}

user_profile = {
    "topic_strength": topic,
    "learning_style": None, 
    "prior_A": None
}

In [ ]:
"""
Summary:
Handles the main user interaction flow for initializing personalization.
This section greets the user, presents topic- and style-based questions, and collects their responses through a command-line interface (CLI).

Remarks:
- The user is prompted to rate their comfort level (1–5) across major CS topics.
- `initial_topic_questions` corresponds directly to `initial_topic_answers`,capturing numeric self-assessments for each topic.
- The personalization questions determine the user’s preferred learning style(e.g., concise vs. detailed, checklist vs. conversational format).
- `flush=True` ensures each prompt appears immediately in the terminal, improving CLI responsiveness during user input.
"""

print("Welcome! Let's personalize your CS study notes!Please answer the following questions.")

initial_topic_questions =["How comfortable are you with Arrays?",
                     "How comfortable are you with Dictionary?",
                     "How comfortable are you with Linked List?",
                     "How comfortable are you with Tree?",
                     "How comfortable are you with Graph?"]
initial_topic_answers = []

initial_personalization_questions = [
    "Do you prefer direct steps or detailed discussion? Press 1 for direct steps, 2 for detailed discussion",
    "Would you prefer a checklist or conversation? Press 1 for checklist, 2 for conversation"
]
initial_personalization_answers = []

for i in range(len(initial_topic_questions)):
    print(initial_topic_questions[i], flush = True)
    initial_topic_answers.append(input("Give answer between 1-5"))

for j in range(len(initial_personalization_questions)):
    print(initial_personalization_questions[j],flush = True)
    initial_personalization_answers.append(input())
    

Welcome! Let's personalize your CS study notes!Please answer the following questions.
How comfortable are you with Arrays?
How comfortable are you with Dictionary?
How comfortable are you with Linked List?
How comfortable are you with Tree?
How comfortable are you with Graph?
Do you prefer direct steps or detailed discussion? Press 1 for direct steps, 2 for detailed discussion
Would you prefer a checklist or conversation? Press 1 for checklist, 2 for conversation


In [ ]:
"""
Summary:
Maps user-provided comfort ratings to topic strengths and classifies learning style.
This section finalizes the initial setup of the `user_profile` dictionary by 
integrating numeric topic strengths and determining the user’s preferred study approach.

Remarks:
- The `zip()` function pairs each topic key with the corresponding comfort rating from `initial_topic_answers`, converting each input value to an integer.
- The learning style is derived from `initial_personalization_answers`:
    • (1,1) → "action-based" (prefers concise, step-driven learning)
    • (2,2) → "relationship-based" (prefers detailed, conversational learning)
    • Mixed responses → "mixed"
- This step completes the first stage of user profiling, enabling adaptive note generation or quiz personalization in later modules.
"""

for key, value in zip(topic, initial_topic_answers):
    topic[key] = int(value)

print(topic)

#Determine learning style
for k in range(len(initial_personalization_answers)-1):
    if initial_personalization_answers[k] == "1" and initial_personalization_answers[k+1] == "1":
        user_profile["learning_style"] = "action-based"
    elif initial_personalization_answers[k] == "2" and initial_personalization_answers[k+1] == "2":
        user_profile["learning_style"] ="relationship-based"
    else:
        user_profile["learning_style"] = "mixed"

print(user_profile)

{'array': 1, 'dictionary': 12, 'linked list': 3, 'tree': 4, 'graph': 5}
{'topic_strength': {'array': 1, 'dictionary': 12, 'linked list': 3, 'tree': 4, 'graph': 5}, 'learning_style': 'action-based', 'prior_A': None}


In [ ]:
"""
Summary:
Identifies the weakest topic from the user's profile based on their self-rated strengths.
This function scans the `topic_strength` dictionary within `user_profile` and returns 
the topic with the lowest numeric score.

Remarks:
- Uses Python’s built-in `min()` with `key=dict.get` to efficiently locate the weakest topic.
- Assumes all topic values are integers representing comfort levels (e.g., 1–5 scale).
- The returned topic string can be used to generate targeted study notes or practice questions.
- Prints the weakest topic immediately for verification during the CLI session.
"""

def find_weakest_topic(user_profile):
    min_key = min(user_profile["topic_strength"], key =user_profile["topic_strength"].get)

    return min_key
print(find_weakest_topic(user_profile))
    

array


In [ ]:
"""
Summary:
Loads API credentials from environment variables and initializes the OpenAI client.
This ensures secure access to the API without hardcoding sensitive information.

Remarks:
- `load_dotenv()` reads key-value pairs from a `.env` file into the environment.
- `os.getenv("OPENAI_API_KEY")` retrieves the API key securely.
- The `OpenAI` client instance is initialized with this key for later use in note or quiz generation.
- Keeping credentials outside the codebase prevents accidental exposure in version control.
"""

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
"""
Summary:
Generates personalized study notes using the OpenAI API based on the user’s weakest topic 
and learning style.

Remarks:
- Builds a dynamic prompt including topic and style preferences.
- Defines formatting rules for action-based, relationship-based, and mixed learners.
- Uses GPT-4o-mini for note generation and prints the result.
"""

full_prompt= f""""
You are an expert Python-based study note generator for computer science students.
Generate concise and helpful study notes on the user's weakest topic: {find_weakest_topic(user_profile)}.
Adapt the tone and structure based on the user's learning style: {user_profile["learning_style"]}. Do not ask for confirmation.

Use the following formatting rules based on the learning style:

If learning_style is "action-based", structure the output as:

- Simple, direct sentences
- Concise bullet points
- Task- and outcome-focused content
- Give structure coding example

If learning_style is "relationship-based", structure the output as:

- Simple, direct sentences
- A narrative or dialogue-style explanation
- Paragraph format with emotional/contextual cues to build understanding
- Give structure coding example

If learning_style is "mixed", structure the output as:
- Combine both bullet points and short explanations
- Blend action-focused clarity with brief contextual insight
- Use small sections with headers like "Concept", "Steps", and "Example"
- Include one short, structured coding example
- Keep tone balanced — slightly instructive yet friendly

Ensure the content remains clear, engaging, and easy to follow regardless of the style.
"""

response = client.chat.completions.create(
        model="gpt-4o-mini",  # or "gpt-3.5-turbo", or "gpt-4o-mini"
        messages=[
            {"role": "user",
            "content": full_prompt},
        ]
    )

# Print the response text
response_text = response.choices[0].message.content
print(response_text)

## Study Notes on Arrays (Action-Based Learning Style)

- **Definition**: An array is a collection of items stored at contiguous memory locations. It can hold multiple values of the same data type.

- **Key Characteristics**:
  - Fixed size.
  - Indexed by integers (starting from 0).
  - Homogeneous data elements.

- **Common Operations**:
  - **Initialization**: Create an array with specified size and data type.
  - **Accessing Elements**: Use index to access specific elements.
  - **Modification**: Update elements using their index.

### Tasks & Outcomes

1. **Initialize an Array**:
   - Task: Create an array of integers.
   - Outcome: Store multiple integer values.
   ```python
   numbers = [1, 2, 3, 4, 5]  # Creating an array of integers
   ```

2. **Access Array Elements**:
   - Task: Get the first element.
   - Outcome: Retrieve a specific value from the array.
   ```python
   first_number = numbers[0]  # Accessing the first element
   print(first_number)  # Output: 1
   ```

3. 

In [ ]:
"""
Summary:
Initializes the embedding model and semantic text splitter for content processing.

Remarks:
- `OpenAIEmbedding()` creates vector embeddings for semantic understanding.
- `SemanticSplitterNodeParser` divides text into meaningful chunks based on 
  embedding similarity, using a buffer for context retention.
"""

embed_model = OpenAIEmbedding()
splitter = SemanticSplitterNodeParser(
    buffer_size=5, breakpoint_percentile_threshold=30, embed_model=embed_model
)

In [ ]:
"""
Summary:
Converts the generated study notes into a document and splits it into semantic nodes.

Remarks:
- Wraps the `response_text` in a `Document` object.
- `splitter.get_nodes_from_documents()` breaks the text into context-aware chunks 
  for later retrieval or vector indexing.
"""

doc = Document(text=response_text)
nodes = splitter.get_nodes_from_documents([doc]) 

In [ ]:
"""
Summary:
Configures a recursive text splitter to divide long text into manageable chunks.

Remarks:
- Splits text into 500-character segments with 20-character overlap.
- Ensures smoother context flow between chunks for embedding or retrieval tasks.
"""

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

In [ ]:
"""
Summary:
Applies the text splitter to the generated notes and inspects the output.

Remarks:
- Creates document chunks from `response_text` for further processing or storage.
- Prints the third chunk for verification and checks total chunk count.
"""
texts = text_splitter.create_documents([response_text])
print(texts[2])
len(texts)

page_content='2. **Access Array Elements**:
   - Task: Get the first element.
   - Outcome: Retrieve a specific value from the array.
   ```python
   first_number = numbers[0]  # Accessing the first element
   print(first_number)  # Output: 1
   ```

3. **Modify an Element**:
   - Task: Change the third element.
   - Outcome: Update the value in the array.
   ```python
   numbers[2] = 10  # Changing the third element
   print(numbers)   # Output: [1, 2, 10, 4, 5]
   ```'


5

In [ ]:
"""
Summary:
Creates and loads FAISS vector indexes for semantic search.

Remarks:
- Generates embeddings using `text-embedding-3-large`.
- Builds a FAISS index from the document chunks for similarity-based retrieval.
- Loads a saved index (`faiss_index_relationship_chunks`) for reuse or comparison.
"""

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
library = FAISS.from_documents(texts, embeddings)
r_chunk_saved = FAISS.load_local("faiss_index_relationship_chunks", embeddings, allow_dangerous_deserialization=True)

In [ ]:
"""
Summary:
Generates multiple-choice quiz questions from each saved FAISS text chunk.

Remarks:
- Iterates through all chunks in the FAISS index and extracts their content.
- Builds a detailed prompt for GPT to create 2–5 programming-related MCQs per chunk.
- Ensures questions test concepts, not analogies, and include correct answers with explanations.
- Appends each generated quiz to `response_text` for review or storage.
"""

for faiss_id in range(r_chunk_saved.index.ntotal):
    # Get the docstore ID
    docstore_id = r_chunk_saved.index_to_docstore_id[faiss_id]
    doc = r_chunk_saved.docstore._dict[docstore_id]
    chunk_text = doc.page_content

    # Prompt for quiz generation
    prompt = f"""
    **DO NOT REPEAT ANY PART OF THIS TEXT CHUNK** in your response.
    Based on the this text chunk {chunk_text}, generate 2-5 multiple choice questions.
    with answers and explanations. Minimum 2 questions, maximum 5 based on relavance. 
    
    - Do not focus on the analogy. Make questions based on programming concept present in the chunk.
    - If you're asking coding based question, make sure to give the relavant code before asking. For example, if you ask what is the output of fruits[0] then surely give fruits array. 
    - Don't make the answer choices obvious. Focus on programming concepts.  
    
    For each question, provide the correct answer immediately after the question, labeled with "ANSWER:", and then provide the explanation labeled with "EXPLANATION:".

    Example format:
    1. What is the output of the following code?
    A) Option A
    B) Option B
    C) Option C
    D) Option D

    ANSWER: B

    EXPLANATION: This is why B is the correct answer.

    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a Python language based Computer Science quiz generator."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )

    # Print the result
    response_text += f"\n\nChunk {faiss_id}\n\n"
    response_text += response.choices[0].message.content
print(response_text)

In [ ]:
"""
Summary:
Parses the generated quiz text into a structured nested dictionary format.

Remarks:
- Iterates through `response_text` line by line to identify chunks, questions, options, answers, and explanations.
- Maintains parsing state with flags (`inside_code`) and trackers (`current_chunk_id`, `current_question_id`).
- Handles fenced code blocks and preserves formatting for readability.
- Builds `outer_dictionary` where each chunk contains multiple questions with structured fields:
  question, options, answer, explanation, and user_answer.
"""

outer_dictionary = {}
current_chunk_id = None
current_question_id = None
inside_code = False

def is_option(s): 
    return bool(re.match(r'^\s*[A-E]\)\s', s))

def normalize_answer(ans):
    m = re.search(r'([A-Ea-e])', ans)
    return m.group(1).upper() if m else ans.strip()

def ensure_chunk():
    global current_chunk_id
    if not current_chunk_id:
        current_chunk_id = "Chunk 0"
        outer_dictionary[current_chunk_id] = {}

def start_question(qnum, first_line=""):
    return {
        "question": first_line.strip(),
        "options": [],
        "answer": None,
        "explanation": None,
        "user_answer": None
    }

for raw in response_text.splitlines():
    line = raw.rstrip("\n")  # keep original spacing for code blocks

    # Chunk header
    if re.match(r'^\s*Chunk\s+\d+\s*$', line):
        current_chunk_id = line.strip()
        outer_dictionary[current_chunk_id] = {}
        current_question_id = None
        inside_code = False
        continue

    # Toggle code fence state AND record the fence in the question
    if line.strip().startswith("```"):
        ensure_chunk()
        if current_question_id:
            q = outer_dictionary[current_chunk_id][current_question_id]["question"]
            outer_dictionary[current_chunk_id][current_question_id]["question"] = (q + ("\n" if q else "") + line)
        inside_code = not inside_code
        continue

    # Question start: "N. <text>"
    m_q = re.match(r'^\s*(\d+)\.\s*(.*)$', line)
    if m_q:
        ensure_chunk()
        qnum, tail = m_q.group(1), m_q.group(2)
        outer_dictionary[current_chunk_id][qnum] = start_question(qnum, tail)
        current_question_id = qnum
        inside_code = False
        continue

    # Options
    if is_option(line):
        if current_chunk_id and current_question_id:
            outer_dictionary[current_chunk_id][current_question_id]["options"].append(line.strip())
        continue

    # ANSWER:
    if line.strip().startswith("ANSWER:"):
        if current_chunk_id and current_question_id:
            ans = line.split(":", 1)[1]
            outer_dictionary[current_chunk_id][current_question_id]["answer"] = normalize_answer(ans)
        continue

    # EXPLANATION:
    if line.strip().startswith("EXPLANATION:"):
        if current_chunk_id and current_question_id:
            text = line.split(":", 1)[1].strip()
            outer_dictionary[current_chunk_id][current_question_id]["explanation"] = text
        continue

    # Append any non-marker line (including code lines and blanks) to the current question
    if current_chunk_id and current_question_id:
        if inside_code or (
            not is_option(line)
            and not line.strip().startswith(("ANSWER:", "EXPLANATION:", "Chunk"))
        ):
            q = outer_dictionary[current_chunk_id][current_question_id]["question"]
            # preserve spacing/blank lines so fenced code renders correctly
            outer_dictionary[current_chunk_id][current_question_id]["question"] = (q + ("\n" if q else "") + line)

# Trim question text
for ch_id, qs in outer_dictionary.items():
    for qid, obj in qs.items():
        if obj.get("question") is not None:
            obj["question"] = obj["question"].strip()

print(outer_dictionary)

{'Chunk 0': {'1': {'question': 'Given the following array declaration in Python:\n```python\nnumbers = [10, 20, 30, 40, 50]\n```\nWhat is the output of the expression `numbers[2]`?', 'options': ['A) 10', 'B) 20', 'C) 30', 'D) 40'], 'answer': 'C', 'explanation': 'In Python, array indexing starts from 0. Therefore, `numbers[2]` refers to the third element of the array, which is 30.', 'user_answer': None}, '2': {'question': 'Which of the following statements is true regarding arrays?', 'options': ['A) Arrays can hold multiple data types.', 'B) The size of an array can be changed dynamically.', 'C) Arrays are stored at non-contiguous memory locations.', 'D) Arrays are indexed starting from 0.'], 'answer': 'D', 'explanation': 'Arrays are indexed by integers starting from 0 in most programming languages. The other statements are incorrect; arrays hold homogeneous data types, have a fixed size, and are stored at contiguous memory locations.', 'user_answer': None}, '3': {'question': 'Consider 

In [ ]:
"""
Summary:
Runs an interactive quiz loop and tracks user performance.

Remarks:
- Displays each question and options, then collects user input.
- Compares user answers with correct ones to count correct and wrong responses.
- Stores incorrectly answered questions in `wrong_question_bank` by chunk ID.
- Provides a foundation for adaptive review or Bayesian performance updates.
"""

correct_count = 0
wrong_count = 0
wrong_question_bank = defaultdict(list)

for outer_key,outer_value in outer_dictionary.items():
    for inner_key,inner_value in outer_value.items():
        print(f"""{inner_key}.{inner_value["question"]}""", flush=True)
        for i in inner_value["options"]:
            print(i, flush=True)
        inner_value["user_answer"] = input("My answer: ")
        if inner_value["answer"] == inner_value["user_answer"]:
            correct_count += 1
        else:
            #We have question and chunk number associated with it here.
            wrong_question_bank[outer_key].append(inner_value["question"])
            wrong_count += 1
                
print(wrong_question_bank)

In [ ]:
"""
Summary:
Prints all incorrectly answered questions for review with total number of correct answers and wrong answers.

Remarks:
- Iterates through `wrong_question_bank` and displays each stored question list.
- Useful for post-quiz analysis or targeted re-learning.
"""

for value in wrong_question_bank.values():
    print(value)

print(wrong_count)
print(correct_count)

['Given the following array declaration in Python:\n```python\nnumbers = [10, 20, 30, 40, 50]\n```\nWhat is the output of the expression `numbers[2]`?', 'Which of the following statements is true regarding arrays?', 'Consider the following code snippet:\n```python\nfruits = ["apple", "banana", "cherry"]\nfruits[1] = "orange"\n```\nWhat will be the content of the `fruits` array after this operation?', 'What will happen if you try to access an index that is out of range for the following array?\n```python\ncolors = ["red", "green", "blue"]\nprint(colors[3])\n```']
['What will be the output of the following code?\n   ```python\n   arr = [10, 20, 30, 40, 50]\n   print(arr[2])\n   ```', 'Given the following code snippet, what will happen if we execute `arr[1] = 100`?\n   ```python\n   arr = [5, 10, 15]\n   arr[1] = 100\n   print(arr)\n   ```', "What will be the result of the following code?\n   ```python\n   fruits = ['apple', 'banana', 'cherry']\n   fruits[1] = 'orange'\n   print(fruits)\n

In [ ]:
"""
Summary:
Computes the posterior probability of topic mastery using Bayes’ theorem.

Remarks:
- `prior`: initial belief about mastery before observing quiz results.
- `n`: total number of questions attempted; `k`: number answered correctly.
- `p_correct_mastered` and `p_correct_not_mastered` represent success likelihoods
  for mastered vs. unmastered states.
- Returns updated mastery probability after incorporating new evidence.
"""

def bayes_mastery_posterior(prior,n, k,  p_correct_mastered=0.9, p_correct_not_mastered=0.25):
 
    p_data_given_M = (p_correct_mastered ** k) * ((1 - p_correct_mastered) ** (n - k))
    p_data_given_notM = (p_correct_not_mastered ** k) * ((1 - p_correct_not_mastered) ** (n - k))
    
    numerator = p_data_given_M * prior
    denominator = numerator + p_data_given_notM * (1 - prior)
    
    posterior = numerator / denominator if denominator > 0 else 0
    return posterior

In [ ]:
"""
Summary:
Sets the initial prior belief of mastery to 0.5.

Remarks:
- Represents equal uncertainty — user is assumed 50% likely to have mastered the topic before any quiz data.
"""

prior = 0.5 # Use only for the first attempt

In [ ]:
"""
Summary:
Defines quiz outcome parameters for Bayesian update.

Remarks:
- `n`: total number of attempted questions.
- `k`: number of correctly answered questions.
"""
n= wrong_count+correct_count
k= correct_count

In [ ]:
"""
Summary:
Updates mastery belief using the Bayesian posterior.

Remarks:
- Calls `bayes_mastery_posterior()` with quiz results to compute the updated belief.
- Reassigns posterior as new prior for future learning sessions.
- Prints the updated mastery probability; may be very low when few answers are correct.
"""

posterior = bayes_mastery_posterior(prior, n, k)
prior = posterior
print(posterior)


1.4316555603695362e-07


In [ ]:
"""
Summary:
Runs a post-note survey to capture user feedback and infer learning style signals.

Remarks:
- Asks 4 questions; stores 0-indexed responses in `survey_responses`.
- Uses `SCORING_MAP` on Q2–Q4 to compute `style_score_from_feedback` (action ↔ relationship).
- Saves Q1’s selection separately (`q1_response_index`) for later Bayesian updates.
- Infers a rough `inferred_style` from the aggregated score and reports the note’s actual style.
"""


survey = [
    {
        "id": 1,
        "question": "How helpful was this note?",
        "options": ["Very helpful and clear", "Somewhat helpful and could be clearer", "Not helpful"]
    },
    {
        "id": 2,
        "question": "How would you describe this note?",
        "options": ["Straight forward and to the point", "Detailed and storylike"]
    },
    {
        "id": 3,
        "question": "What would you prefer more in this note?",
        "options": ["More step by step instructions", "More background, context, stories"]
    },
    {
        "id": 4,
        "question": "Overall does this note match your learning style?",
        "options": ["Perfect match", "Needs more details, stories, example", "Needs more concise example"]
    }
]

# Scores based on 0-indexed option: Positive = Action-Based, Negative = Relationship-Based
# Q1 will be used as the Likelihood, not in this map.
SCORING_MAP = {
    2: {0: 2, 1: -2}, # Q2: "Straight forward" (+2), "Detailed and storylike" (-2)
    3: {0: 2, 1: -2}, # Q3: "Step by step" (+2), "Background, context" (-2)
    4: {0: 0, 1: -2, 2: 2} # Q4: "Concise example" (+2), "Details, stories, example" (-2)
}

survey_responses = {}
q1_response_index = None
style_score_from_feedback = 0

print("\n--- Note Feedback Survey ---")

for item in survey:
    q_id = item["id"]
    print(f"\nQuestion {q_id}: {item['question']}")
    for j in range(len(item["options"])):
        print(f"{j+1}. {item['options'][j]}")

    while True:
        try:
            # Get user input and convert to 0-based index
            user_input = input("Press only 1 desired number: ")
            response_index = int(user_input) - 1 
            
            if 0 <= response_index < len(item["options"]):
                survey_responses[q_id] = response_index
                
                # Calculate Style Score from Q2, Q3, Q4
                if q_id in SCORING_MAP:
                    style_score_from_feedback += SCORING_MAP[q_id].get(response_index, 0)
                
                # Store Q1 response for Bayesian update
                if q_id == 1:
                    q1_response_index = response_index
                    
                break 
            else:
                print(f"Invalid input. Please enter a number between 1 and {len(item['options'])}.")
        except ValueError:
            print("Invalid input. Please enter a numerical digit.")

print("\nSurvey responses recorded.")

# This classification is based only on Q2, Q3, Q4 for a rough estimate
if style_score_from_feedback > 2:
    inferred_style = "action-based"
elif style_score_from_feedback < -2:
    inferred_style = "relationship-based"
else:
    inferred_style = "mixed"

note_style = user_profile["learning_style"] 
print(f"The note provided was designed for a: {note_style} learner.")


--- Note Feedback Survey ---

Question 1: How helpful was this note?
1. Very helpful and clear
2. Somewhat helpful and could be clearer
3. Not helpful

Question 2: How would you describe this note?
1. Straight forward and to the point
2. Detailed and storylike

Question 3: What would you prefer more in this note?
1. More step by step instructions
2. More background, context, stories

Question 4: Overall does this note match your learning style?
1. Perfect match
2. Needs more details, stories, example
3. Needs more concise example

Survey responses recorded.
The note provided was designed for a: relationship-based learner.


In [ ]:
"""
Summary:
Updates the probability the user is action-based using Bayesian evidence from survey feedback.

Remarks:
- Sets a prior from Q2–Q4 (inferred_style), then updates with Q1 helpfulness vs note_style.
- Handles 'action-based', 'relationship-based', and 'mixed' note styles with sensible likelihoods.
- Returns posterior P(A); code then maps it to a label and updates user_profile.
- Prints old style, posterior, and the updated style for visibility.
"""


def bayes_style_posterior(prior_A, note_style, q1_response_index, inferred_style, p_helpful_if_match=0.85, p_helpful_if_mismatch=0.15):
    """
    Updates the probability of the user being Action-Based (P(A))
    based on the helpfulness rating (Q1) of the note (Evidence).
    
    :param prior_A: P(A) - Prior probability of being Action-Based (will be overwritten by inferred_style).
    :param note_style: The style of the note presented ('action-based', 'relationship-based', or 'mixed').
    :param q1_response_index: 0-indexed response to Q1 (0=Very helpful, 1=Somewhat, 2=Not helpful).
    :param inferred_style: Initial style inferred from Q2, Q3, Q4.
    :param p_helpful_if_match: P(Helpful | Match) - Probability of rating 'Very Helpful' if styles match.
    :param p_helpful_if_mismatch: P(Helpful | Mismatch) - Probability of rating 'Very Helpful' if styles mismatch.
    """
    
    if inferred_style == "action-based":
        current_prior_A = 0.7 
    elif inferred_style == "relationship-based":
        current_prior_A = 0.3
    else:
        current_prior_A = 0.5 # Neutral prior for mixed or initial run
        
    prior_R = 1 - current_prior_A
    
    is_helpful = (q1_response_index == 0)
    is_not_helpful = (q1_response_index == 2)
    
    # If the note was Action-Based:
    if note_style == 'action-based':
        if is_helpful:
            p_data_given_A = p_helpful_if_match       # High if style matches and helpful
            p_data_given_R = 1 - p_helpful_if_match  # Low if style mismatches but still helpful
        elif is_not_helpful:
            p_data_given_A = 1 - p_helpful_if_match  # Low if style matches but NOT helpful
            p_data_given_R = p_helpful_if_match      # High if style mismatches and NOT helpful
        else: # Somewhat helpful (Neutral Evidence)
            p_data_given_A = 0.5
            p_data_given_R = 0.5
            
    # If the note was Relationship-Based:
    elif note_style == 'relationship-based':
        # P(D | A): If 'A' is true, how likely is the response? (MISMATCH)
        if is_helpful:
            p_data_given_A = 1 - p_helpful_if_match  # Low if style mismatches but helpful
            p_data_given_R = p_helpful_if_match      # High if style matches and helpful
        elif is_not_helpful:
            p_data_given_A = p_helpful_if_match      # High if style mismatches and NOT helpful
            p_data_given_R = 1 - p_helpful_if_match  # Low if style matches but NOT helpful
        else: # Somewhat helpful (Neutral Evidence)
            p_data_given_A = 0.5
            p_data_given_R = 0.5
    
    #If the note was Mixed-Based:
    elif note_style == 'mixed':
        # A mixed note is a reasonable match for both A and R learners.
        if is_helpful:
            # Helpful: Suggests *any* specific style is *more* likely than a strong mismatch.
            # We'll use a slightly lower likelihood than a perfect match, but higher than 0.5
            p_data_given_A = 0.7  # P(D | A)
            p_data_given_R = 0.7  # P(D | R)
        elif is_not_helpful:
            # Not Helpful: Strong rejection, despite the note being mixed. Leads to high uncertainty.
            p_data_given_A = 0.5  # Neutral evidence
            p_data_given_R = 0.5  # Neutral evidence
        else: # Somewhat helpful (Neutral Evidence)
            p_data_given_A = 0.5
            p_data_given_R = 0.5
    
    numerator = p_data_given_A * current_prior_A
    denominator = numerator + p_data_given_R * prior_R
    
    posterior_A = numerator / denominator if denominator > 0 else current_prior_A
    
    return posterior_A


current_note_style = user_profile.get("learning_style", "action-based") 

# Initialize prior once
if "prior_A" not in user_profile:
    user_profile["prior_A"] = 0.5

posterior_A = bayes_style_posterior(
    prior_A=user_profile["prior_A"],
    note_style=current_note_style,
    q1_response_index=q1_response_index,
    inferred_style=inferred_style
)

user_profile["prior_A"] = posterior_A

if posterior_A > 0.65:
    new_style = "action-based"
elif posterior_A < 0.35:
    new_style = "relationship-based"
else:
    new_style = "mixed"

user_profile["learning_style"] = new_style

print("\n" + "="*40)
print(f"| Old Style: {current_note_style:<28} |")
print(f"| New P(Action-Based): {posterior_A:.3f}{' ':<20} |")
print(f"| Updated Learning Style: {user_profile['learning_style']:<19} |")
print("="*40)


| Old Style: relationship-based           |
| New P(Action-Based): 0.850                     |
| Updated Learning Style: action-based        |


In [ ]:
"""
Summary:
Generates follow-up study notes using posterior mastery and learning style.

Remarks:
- Tailors depth by posterior:
  <0.3 → beginner notes, ~0.6 → review wrong topics from `wrong_question_bank`, >0.9 → advanced material.
- Mentions specific missed questions/subtopics; includes code when referencing programming questions.
- Adapts format to `user_profile["learning_style"]` (action-based, relationship-based, mixed).
- Sends the composed prompt to GPT-4o-mini and prints the response.
"""

full_prompt= f""""
You are an expert Python-based study note generator for computer science students.
Student took a MCQ test on topic {find_weakest_topic(user_profile)} and performed {posterior}.
If posterior is <0.3 it means user doesn't know anything about the topic so give easy to read and clear, concise notes again. 
If posterior is ~0.6 it means user partially knows about the topic so review the wrong questions {wrong_question_bank} and focus on those topics.
If posterior is >0.9 it means user knows about the topic so proceed to advanced learning of this topic.

Also, based on posterior discuss which questions and sub topics user got wrong. If you are discussing specific questions from programming questions, must mention code associated with the question.

Adapt the tone and structure based on the user's learning style: {user_profile["learning_style"]}.

Use the following formatting rules based on the learning style:

If learning_style is "action-based", structure the output as:

- Simple, direct sentences
- Concise bullet points
- Task- and outcome-focused content
- Give structure coding example

If learning_style is "relationship-based", structure the output as:

- Simple, direct sentences
- A narrative or dialogue-style explanation
- Paragraph format with emotional/contextual cues to build understanding
- Give structure coding example

If learning_style is "mixed", structure the output as:
- Combine both bullet points and short explanations
- Blend action-focused clarity with brief contextual insight
- Use small sections with headers like "Concept", "Steps", and "Example"
- Include one short, structured coding example
- Keep tone balanced — slightly instructive yet friendly

Ensure the content remains clear, engaging, and easy to follow regardless of the style.
"""


response = client.chat.completions.create(
        model="gpt-4o-mini",  # or "gpt-3.5-turbo", or "gpt-4o-mini"
        messages=[
            {"role": "user",
            "content": full_prompt},
        ]
    )

    # Print the response text
response_text = response.choices[0].message.content
print(response_text)

Given your performance on the topic of arrays, we'll review the necessary concepts and address the questions you've struggled with. Here’s a structured approach to enhance your understanding. 

### Topic: Arrays in Python

#### Key Concepts to Review:

- **Array Definition**: 
  - An array is a data structure that holds a fixed-size sequential collection of elements of the same type.

- **Common Operations**:
  - **Accessing Elements**: 
    - Use an index: `my_array[index]`
  - **Updating Elements**: 
    - Assign a new value to an index: `my_array[index] = new_value`
  - **Adding Elements**:
    - **Append**: `my_array.append(value)` - adds to the end of the array.
    - **Insert**: `my_array.insert(index, value)` - adds at specified position.
  - **Removing Elements**:
    - **Remove**: `my_array.remove(value)` - removes the first occurrence of a value.
    - **Pop**: `my_array.pop(index)` - removes and returns element at specified index.

#### Specific Questions to Focus On:

You p